### Чому можна довіряти динаміці ref для оцінки реального тренду low

**ref** і **low** — той самий модельний ряд датчика (v20), стоять в одній грядці, на відстані ~2 см один від одного. Раніше, під час ручного калібрування (з відром, вдома), було видно: абсолютні значення можуть відрізнятись — ґрунт неоднорідний, десь щільніше, десь слабше — але це різниця рівня, не різниця поведінки.

Обидва датчики незалежно реагують на ту саму зовнішню причину — полив (20 хв тому, 5 год тому, 20 год тому) — і мають показати однакову динаміку зміни в часі, навіть якщо самі числа різні.

Тому для питання "як швидко реально змінюється вологість за 15 хв / за 4 год" можна законно скористатись щільним потоком ref як проксі динаміки — хоча порівнювати абсолютний рівень чи шум **ref** і **low** напряму, як ми вже з'ясували раніше, некоректно.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

path = Path("data/measurement.csv")
df = pd.read_csv(path, parse_dates=["timestamp"])
ref = df[(df["node_id"] == "stage3-reference") & (df["timestamp"] >= "2026-08-25")]
ref = ref.sort_values(by="timestamp")

print(ref.shape)

(24829, 11)


In [2]:
ref_indexed = ref.set_index("timestamp")
ref_15min = ref_indexed["soil_raw"].resample("15min").mean()

In [3]:
print(ref_15min.head(10))
print(ref_15min.shape)

timestamp
2026-08-25 00:00:00    1282.517647
2026-08-25 00:15:00    1282.827830
2026-08-25 00:30:00    1283.645540
2026-08-25 00:45:00    1284.023529
2026-08-25 01:00:00    1283.296471
2026-08-25 01:15:00    1280.863850
2026-08-25 01:30:00    1281.948478
2026-08-25 01:45:00    1281.698824
2026-08-25 02:00:00    1280.329177
2026-08-25 02:15:00    1280.234742
Freq: 15min, Name: soil_raw, dtype: float64
(86,)


In [4]:
change_15min = ref_15min.diff().abs()
change_4h = ref_15min.diff(16).abs()

print("15 хв:", change_15min.median(), change_15min.max())
print("4 год:", change_4h.median(), change_4h.max())

15 хв: 1.3772021378488262 10.527563341251152
4 год: 20.48479726603307 36.93533321357995


### Висновок: усереднення в межах бурста безпечне

За 15 хвилин ґрунт реально змінюється в середньому на 1.38 одиниці ADC — **менше, ніж наш власний шум** усередині усталеної частини бурста (1.74). На масштабі одного циклу сну (тим більше — одного бурста, 19.5с) реальний сигнал практично не рухається.

Значить, коли ми усереднюємо позиції 2-6 всередині одного бурста, ми не "розмазуємо" якийсь швидкий реальний процес — там просто нічому мінятись за ці секунди. Усереднення чисто прибирає шум, не втрачає інформацію.

Для порівняння: за 4 години ґрунт змінюється в середньому на 20.48 — практично стільки ж, скільки коштує неправильно врахований стрибок прогріву (~20-23). Це і є справжня ціна помилки позиції 0: вона еквівалентна ~4 годинам "застарілих" даних.